# Olist Delivery Performance Analysis

## tl;dr

- Across **96,281** eligible delivered orders, the late rate is **8.1%**.
- Among classified late orders, **27.2%** involve handoff after the seller shipping limit, while **72.8%** were handed off on time and are localized post-handoff.
- The average review score for on-time orders is **1.73 points** higher than for late orders; the association is consistent across distance bands.
- The 90-day repeat-order gap is only **0.3%** and is unstable across distance bands, so it is not used as the main business case.
- Recommendation: pilot seller deadline alerts and route-SLA monitoring in segments with the highest excess late orders.

## Context & Methods

### Business decision

Identify where delays are localized and which sellers and routes should be prioritized for an OTD improvement pilot.

### Key assumptions

- Grain: one row per `order_id`.
- Late means actual delivery occurred after the promised date.
- Seller-stage SLA is analyzed only for single-seller orders.
- Review and repeat-purchase results are associations, not proven causal effects.
- The monetary metric is order value at risk, not realized loss.

All calculation logic is stored in `sql/` and `scripts/`; the notebook runs the same saved queries.

## Data

### 1. Connect to the analytical database

In [1]:
from pathlib import Path
import re
import sqlite3
import pandas as pd
from IPython.display import display

project_dir = Path.cwd().parent
database_path = project_dir / 'data' / 'processed' / 'olist_supply_chain.sqlite'
query_path = project_dir / 'sql' / '02_analysis_queries.sql'
output_dir = project_dir / 'outputs' / 'tables'
connection = sqlite3.connect(database_path)
database_path

WindowsPath('data/processed/olist_supply_chain.sqlite')

### 2. Parse named SQL queries

In [2]:
sql_text = query_path.read_text(encoding='utf-8')
pattern = re.compile(r'^-- name: ([a-zA-Z0-9_]+)\s*$', re.MULTILINE)
matches = list(pattern.finditer(sql_text))
queries = {}
for index, match in enumerate(matches):
    start = match.end()
    end = matches[index + 1].start() if index + 1 < len(matches) else len(sql_text)
    queries[match.group(1)] = sql_text[start:end].strip().rstrip(';')
sorted(queries)

['distance_bands',
 'monthly_trend',
 'overall_kpis',
 'priority_categories',
 'priority_routes',
 'priority_sellers',
 'repeat_90d',
 'repeat_90d_by_distance',
 'review_impact',
 'review_impact_by_distance',
 'spike_distance_decomposition',
 'spike_stage_mix',
 'stage_attribution']

## Results

### 3. Baseline KPI

In [3]:
overall = pd.read_sql_query(queries['overall_kpis'], connection)
overall.T

,0
eligible_orders,9.628100e+04
late_orders,7.822000e+03
on_time_rate,9.187586e-01
late_rate,8.124137e-02
avg_late_days,9.554596e+00
gross_order_value,1.539068e+07
late_order_value,1.351183e+06
late_freight_value,1.926180e+05
on_time_review_score,4.294544e+00
late_review_score,2.565887e+00


Late rate is calculated only for delivered orders with complete and chronologically valid dates.

### 4. Establish the time pattern

In [4]:
monthly = pd.read_sql_query(queries['monthly_trend'], connection)
monthly.sort_values('late_rate', ascending=False).head(8)

,purchase_month,orders,late_orders,late_rate,avg_days_vs_promise
14,2018-03,7003,1496,0.213623,-5.731657
13,2018-02,6555,1048,0.159878,-7.583858
10,2017-11,7288,1043,0.143112,-7.399620
11,2017-12,5513,462,0.083802,-12.286421
16,2018-05,6719,556,0.082750,-11.453160
3,2017-04,2303,181,0.078593,-12.431898
12,2018-01,7069,464,0.065639,-12.221993
2,2017-03,2545,142,0.055796,-11.780223


In [5]:
spike_detail = pd.read_csv(output_dir / 'spike_distance_decomposition_detail.csv')
spike_detail[['distance_band', 'orders', 'late_orders', 'late_rate', 'baseline_late_rate', 'expected_late_at_band_baseline']]

,distance_band,orders,late_orders,late_rate,baseline_late_rate,expected_late_at_band_baseline
0,1500+ km,1841,500,0.271592,0.098718,181.739379
1,250-749 km,9577,1858,0.194006,0.046817,448.366180
2,750-1499 km,3821,755,0.197592,0.056478,215.802968
3,<250 km,5498,457,0.083121,0.040455,222.421535
4,unknown,109,17,0.155963,0.088496,9.646018


The mix-decomposition check shows that peak months cannot be explained by a larger share of long-distance deliveries: late rate increased within distance bands. The dataset does not contain the operational event that caused the change.

### 5. Localize the operational stage

In [6]:
stage_attribution = pd.read_sql_query(queries['stage_attribution'], connection)
stage_attribution

,stage_bucket,orders,order_value,avg_review_score,avg_seller_stage_days,avg_carrier_stage_days
0,on_time,88459,14039501.12,4.294544,3.003115,7.889079
1,late_downstream_candidate,5680,920326.20,2.583648,3.051931,28.931035
2,late_seller_stage_involved,2124,426080.72,2.522892,13.296200,17.037942
3,late_unclassified_multi_seller,18,4775.73,2.000000,4.064889,21.700737


`late_seller_stage_involved` means handoff occurred after the shipping limit. `late_downstream_candidate` means the seller deadline was met but the order was still delivered late. This localizes the issue; it does not prove seller or carrier fault.

### 6. Diagnose distance and concentration

In [7]:
distance = pd.read_sql_query(queries['distance_bands'], connection)
distance

,distance_band,orders,late_orders,late_rate,avg_carrier_stage_days,avg_review_score,order_value
0,<250 km,27269,1719,0.063039,4.348637,4.276866,3680611.41
1,250-749 km,42451,3393,0.079927,9.664692,4.130713,6692945.54
2,750-1499 km,17323,1511,0.087225,12.360294,4.104211,3047813.26
3,1500+ km,8762,1149,0.131134,17.155127,4.003683,1896533.26
4,unknown,476,50,0.105042,11.508631,4.195789,72780.30


In [8]:
priority_routes = pd.read_sql_query(queries['priority_routes'], connection)
priority_routes.head(10)

,route,seller_state,customer_state,orders,late_orders,late_rate,avg_distance_km,order_value,late_order_value,avg_review_score,baseline_late_rate,excess_late_orders
0,SP -> RJ,SP,RJ,8140,1264,0.155283,440.873019,1253994.29,197395.04,3.887184,0.081241,602.695277
1,SP -> BA,SP,BA,2306,345,0.149610,1390.155988,375322.77,59748.69,3.870306,0.081241,157.657409
2,SP -> ES,SP,ES,1454,207,0.142366,786.697238,216225.40,29421.85,4.050035,0.081241,88.875053
3,SP -> CE,SP,CE,969,148,0.152735,2258.288104,183830.14,27417.67,3.964212,0.081241,69.277116
4,SP -> SC,SP,SC,2317,256,0.110488,561.441145,357622.70,35567.18,4.034828,0.081241,67.763754
5,SP -> MA,SP,MA,489,104,0.212679,2178.288121,88067.41,22528.94,3.716495,0.081241,64.272972
6,PR -> RJ,PR,RJ,969,135,0.139319,843.322787,172302.06,23158.37,3.888075,0.081241,56.277116
7,SP -> AL,SP,AL,255,67,0.262745,1902.904837,50694.07,11616.78,3.719368,0.081241,46.283452
8,SP -> PA,SP,PA,682,90,0.131965,2326.509063,137073.84,17779.23,3.864785,0.081241,34.593388
9,SP -> PI,SP,PI,327,60,0.183486,2014.666701,63207.61,11708.17,3.956656,0.081241,33.434073


In [9]:
priority_sellers = pd.read_sql_query(queries['priority_sellers'], connection)
priority_sellers.head(10)

,primary_seller_id,seller_state,orders,late_orders,late_rate,order_value,late_order_value,avg_review_score,baseline_late_rate,excess_late_orders
0,06a2c3af7b3aee5d69171b0e14f0ee87,MA,384,90,0.234375,47532.37,10836.56,4.013158,0.081241,58.803315
1,4a3ca9315b744ce9f8e9374361493884,SP,1723,192,0.111434,231068.03,26797.56,3.883803,0.081241,52.021126
2,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1114,130,0.116697,244882.40,29527.43,4.169982,0.081241,39.497118
3,1f50f920176fa81dab994f9023523100,SP,1381,148,0.107169,141386.15,13878.13,4.162064,0.081241,35.805673
4,8160255418d5aaa7dbdc9f4c64ebda44,SP,370,62,0.167568,54496.97,9532.97,3.978142,0.081241,31.940694
5,88460e8ebdecbfecb5f9601833981930,PR,231,48,0.207792,35927.40,7921.06,3.471739,0.081241,29.233244
6,ea8482cd71df3c1969d7b9473ff13abc,SP,1117,118,0.105640,53456.19,5760.92,4.036904,0.081241,27.253394
7,7d13fca15225358621be4086e1eb0964,SP,556,68,0.122302,121699.38,16341.48,4.045208,0.081241,22.829800
8,e5a3438891c0bfdb9394643f95273d8e,SP,213,40,0.187793,10744.28,2049.90,4.018868,0.081241,22.695589
9,8b321bb669392f5163d04c59e235e066,SP,919,96,0.104461,30845.75,3231.19,4.121311,0.081241,21.339184


Priority is ranked by `excess_late_orders`, not late rate alone. This prevents a small high-rate segment from displacing a larger operational problem.

### 7. Quantify customer-experience association

In [10]:
review_impact = pd.read_sql_query(queries['review_impact'], connection)
review_impact

,delivery_group,reviewed_orders,avg_review_score,low_review_rate,five_star_rate
0,on_time,87979,4.294544,0.091806,0.623865
1,late,7657,2.565887,0.540029,0.221888


In [11]:
review_by_distance = pd.read_sql_query(queries['review_impact_by_distance'], connection)
review_by_distance.pivot(index='distance_band', columns='delivery_group', values=['avg_review_score', 'low_review_rate'])

avg_review_score           low_review_rate          
delivery_group             late   on_time            late   on_time
distance_band                                                      
1500+ km               2.339286  4.249967        0.597321  0.094596
250-749 km             2.427733  4.276344        0.578139  0.095558
750-1499 km            2.497290  4.254908        0.554878  0.097846
<250 km                3.043491  4.358893        0.414793  0.081894
unknown                2.700000  4.371765        0.520000  0.068235

In [12]:
statistical_effects = pd.read_csv(output_dir / 'statistical_effects.csv')
statistical_effects

,metric,effect,ci_95_low,ci_95_high,interpretation
0,review_score_on_time_minus_late,1.728656,1.690765,1.766548,"Association, not causal effect"
1,low_review_rate_late_minus_on_time,0.448223,0.436898,0.459548,"Association, not causal effect"
2,repeat_90d_on_time_minus_late,0.003291,0.000859,0.005722,Censored observational association


### 8. Check repeat purchase with censoring

In [13]:
repeat_90d = pd.read_sql_query(queries['repeat_90d'], connection)
repeat_90d

,delivery_group,eligible_orders,repeat_orders,repeat_90d_rate,avg_order_value
0,on_time,78286,1002,0.012799,158.191192
1,late,6836,65,0.009508,174.425557


In [14]:
repeat_by_distance = pd.read_sql_query(queries['repeat_90d_by_distance'], connection)
repeat_by_distance.pivot(index='distance_band', columns='delivery_group', values='repeat_90d_rate')

delivery_group,late,on_time
distance_band,,
1500+ km,0.010214,0.008387
250-749 km,0.008766,0.012801
750-1499 km,0.012221,0.011061
<250 km,0.007073,0.015164
unknown,0.023256,0.013055


The repeat-order gap is small and changes sign in some distance bands. Repeat purchase therefore remains supporting evidence rather than the main argument for the project.

### 9. Size financial scenarios without claiming causality

In [15]:
financial_scenarios = pd.read_csv(output_dir / 'financial_scenarios.csv')
financial_scenarios

,relative_late_reduction,attributable_share_of_repeat_gap,late_orders_reduced,protected_order_value,estimated_incremental_repeat_orders,estimated_incremental_repeat_gmv,scenario_note
0,0.1,0.25,782.2,135118.265,0.643504,93.588029,Directional; not a causal forecast
1,0.1,0.50,782.2,135118.265,1.287008,187.176059,Directional; not a causal forecast
2,0.1,1.00,782.2,135118.265,2.574016,374.352117,Directional; not a causal forecast
3,0.2,0.25,1564.4,270236.530,1.287008,187.176059,Directional; not a causal forecast
4,0.2,0.50,1564.4,270236.530,2.574016,374.352117,Directional; not a causal forecast
5,0.2,1.00,1564.4,270236.530,5.148032,748.704234,Directional; not a causal forecast
6,0.3,0.25,2346.6,405354.795,1.930512,280.764088,Directional; not a causal forecast
7,0.3,0.50,2346.6,405354.795,3.861024,561.528176,Directional; not a causal forecast
8,0.3,1.00,2346.6,405354.795,7.722048,1123.056351,Directional; not a causal forecast


`protected_order_value` is the value of orders that would receive a better delivery outcome under the assumed late-rate reduction. It is not revenue uplift. Repeat GMV is shown only as a sensitivity range.

### 10. Validate calculations and claims

In [16]:
validation_checks = pd.read_csv(output_dir / 'validation_checks.csv')
validation_checks

,check,status,evidence
0,One fact row per raw order,PASS,99441 fact rows / 99441 raw orders
1,Order value reconciles to raw item value plus ...,PASS,difference=0.000000
2,No negative stage durations in eligible popula...,PASS,0 rows
3,Stage buckets reconcile to eligible population,PASS,96281 / 96281
4,Review association direction holds in all know...,PASS,4/4 bands
5,Repeat association direction across distance b...,WARN,2/4 bands


## Takeaways

1. Most classified late orders occur after an on-time seller handoff, but roughly one quarter involve the seller stage.
2. Peak months reflect within-route degradation, not distance mix alone.
3. The review-score association is strong and consistent; the repeat-purchase association is weak and unstable across segments.
4. The first pilot should combine seller deadline alerts with route-SLA monitoring.
5. The public README must preserve the limitations: no carrier ID, internal warehouse events, or causal experiment.

In [17]:
connection.close()
'Connection closed'

'Connection closed'